# Experiment 5: Allen-Zhu-style Paired CFGs — Capacity-Dependent Multilingual ε

**Goal.** Test whether training a single transformer on two independently-generated,
Allen-Zhu-style layered CFGs that share terminal symbols costs more capacity than
training on the same two grammars with disjoint terminal alphabets. The gap between
these two conditions (`Bi-shared` vs `Bi-disjoint`) isolates the representational
conflict cost — this is our operationalization of ε — from the raw computational
load of learning two grammars.

**Why this design (vs. every prior ε attempt on the toy PCFG substrate):**
- The base parsing task is genuinely capacity-demanding by construction (Allen-Zhu
  & Li, *Physics of Language Models Part 1*, showed GPT-2-small is near its
  learnability limit on this CFG family), so capacity actually binds — the recurring
  failure of every prior attempt on the small TinyGPT substrate.
- Shared terminal embeddings force a genuine shared resource (the "semantic hub")
  that two structurally different grammars must compete to use — not a lookup table,
  fires on every token, not bypassable by an architectural shortcut, and both
  languages equally need it (no zero-benefit-of-separation failure mode).
- `Bi-disjoint` is the critical control: same computational load, zero representational
  conflict. The Bi-shared − Bi-disjoint gap is a clean, targeted ε measurement.

**Structure of this notebook:**
1. CFG construction (layered grammar generator, following Allen-Zhu Appendix A.1)
2. Layered CYK-style validity parser
3. Diagnostics: rule-overlap audit, n-gram cross-entropy baseline
4. Model: GPT-2 with rotary position embeddings (RoPE)
5. Data pipeline (mono-A, mono-B, bi-shared, bi-disjoint)
6. Training loop
7. Evaluation (generation accuracy, cross-entropy, ε computation)
8. Pre-checks (run these BEFORE committing to the full sweep — each gates the next)
9. Main sweep orchestration
10. Diagnostics (tag-swap test, cross-grammar KL, per-position loss)
11. Analysis & plotting

**Compute note.** Pre-checks 1-2 run one grammar at full capacity (d_model=768,
~85M params) for up to 100-200K steps — budget for this before running anything else.
The full sweep (4 conditions × 5 capacities × 3 seeds = 60 runs) is the expensive
part; only launch it after all five pre-checks pass. Every design choice below
(grammar size, distinctness constraint, rule count) was empirically checked in a
CPU sandbox before this notebook was written — see the printed diagnostics in each
section for what was verified and why the defaults are what they are.


## 0. Setup

In [1]:

# Self-contained setup cell. Run this first.
import os, sys, math, json, time, random, itertools, contextlib
from collections import defaultdict, Counter
from dataclasses import dataclass, field, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
if str(DEVICE).startswith('cuda'):
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# bf16 autocast for the forward pass. This is a DEVIATION from Allen-Zhu's own
# float32 recipe -- almost certainly safe at this model scale (bf16 has the same
# exponent range as fp32, so no gradient scaler is needed, unlike fp16), but if
# you want to rule out any optimization-dynamics difference before trusting a
# 100-200K-step run to it: train Pre-check 1 for a few thousand steps with
# USE_BF16=True and again with USE_BF16=False and confirm the loss curves track
# closely before committing the full run.
USE_BF16 = (str(DEVICE).startswith('cuda'))
print(f"bf16 autocast: {USE_BF16}")

RESULTS_DIR = "exp5_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "checkpoints"), exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def autocast_ctx():
    '''bf16 autocast on CUDA when USE_BF16 is on; a no-op context otherwise
    (CPU, or USE_BF16=False for the float32-vs-bf16 sanity check above).'''
    if USE_BF16 and str(DEVICE).startswith('cuda'):
        return torch.autocast(device_type='cuda', dtype=torch.bfloat16)
    return contextlib.nullcontext()


Device: cuda
GPU: NVIDIA A100 80GB PCIe
Memory: 85.1 GB
bf16 autocast: True


## 1. Layered CFG Construction

Follows Allen-Zhu & Li, *Physics of Language Models Part 1*, Section 2.1 and
Appendix A.1 (the `cfg3` family): depth-7 layered grammars, 3 nonterminals per
level, 3 terminal symbols, rules of length 2 or 3, uniform rule probabilities.

**Two deviations from the paper, both empirically checked before adopting:**

1. **Distinctness constraint kept on for all difficulty levels** (Allen-Zhu only
   uses it for their easiest variant, `cfg3b`). Testing showed that without it,
   terminal marginal frequencies become highly skewed (unigram entropy 1.39 out of
   a possible 1.58 bits) purely from compounding of the same symbol across 6
   expansion levels — this creates an easy unigram-frequency shortcut unrelated to
   real grammar difficulty. With the constraint, entropy is 1.58/1.58 (over both
   grammars we generate) — genuinely closer to uniform, and the resulting
   grammars are still highly non-trivial (see the n-gram diagnostic below).
2. **`rules_per_nt=3` is the tested default**, not `cfg3f`'s 3-4. We're pairing
   *two* CFGs in the shared-terminal condition, so we deliberately leave headroom
   below GPT-2-small's absolute limit on a *single* grammar (which Allen-Zhu
   showed is `cfg3f`, ~3-4 rules/NT) — Pre-check 1 checks whether this succeeds;
   Pre-check 3 checks whether capacity still binds at all sweep points. Bump to 4
   if Pre-check 3 shows the grammar is too easy (ceiling accuracy even at small
   `d_model`).


In [2]:

def generate_layered_cfg(depth, n_nt, n_t, rules_per_nt, seed, rhs_lengths=(2, 3),
                          distinct_consecutive=True, max_attempts_per_rule=500):
    '''
    Layered CFG. Level 0 = root (1 symbol). Levels 1..depth-2 have n_nt symbols
    each. Level depth-1 is the terminal level (n_t symbols). Every non-terminal
    has `rules_per_nt` distinct RHS rules of length 2 or 3, drawn from the next
    level down.

    Returns a dict:
      levels    : list of lists of symbol ids, one list per level (level 0 = [root])
      rules     : dict symbol -> list of rhs tuples
      terminals : set of terminal symbol ids
      root      : root symbol id
      depth     : depth
    '''
    rng = random.Random(seed)
    levels = []
    next_id = 0
    levels.append([next_id]); next_id += 1
    for _ in range(1, depth - 1):
        levels.append(list(range(next_id, next_id + n_nt))); next_id += n_nt
    terminals = list(range(next_id, next_id + n_t)); next_id += n_t
    levels.append(terminals)

    rules = {}
    for lvl in range(0, depth - 1):
        child_level = levels[lvl + 1]
        for sym in levels[lvl]:
            rhs_set = set()
            attempts = 0
            while len(rhs_set) < rules_per_nt and attempts < max_attempts_per_rule:
                attempts += 1
                L = rng.choice(rhs_lengths)
                rhs = tuple(rng.choice(child_level) for _ in range(L))
                if distinct_consecutive and any(rhs[i] == rhs[i+1] for i in range(len(rhs)-1)):
                    continue
                rhs_set.add(rhs)
            if len(rhs_set) < rules_per_nt:
                raise RuntimeError(
                    f"Could not find {rules_per_nt} distinct rules for symbol {sym} "
                    f"after {max_attempts_per_rule} attempts (child level size {len(child_level)} "
                    f"may be too small for this many distinct-consecutive rules of these lengths)."
                )
            rules[sym] = list(rhs_set)
    return {'levels': levels, 'rules': rules, 'terminals': set(terminals),
            'root': levels[0][0], 'depth': depth}


def sample_string(grammar, rng):
    '''Top-down sample a terminal string from the grammar.'''
    seq = [grammar['root']]
    for _ in range(grammar['depth'] - 1):
        new_seq = []
        for sym in seq:
            rhs = rng.choice(grammar['rules'][sym])
            new_seq.extend(rhs)
        seq = new_seq
    return seq


def remap_terminals(grammar, offset):
    '''Return a copy of `grammar` with terminal symbol ids shifted by `offset`.
    Used to build the Bi-disjoint condition: same grammar B, disjoint alphabet.
    Only the bottom-level rules (whose RHS are terminals) and the terminal set
    need remapping; all NT-level rules are structurally identical.'''
    import copy
    g2 = copy.deepcopy(grammar)
    old_terms = grammar['terminals']
    remap = {t: t + offset for t in old_terms}
    g2['terminals'] = set(remap.values())
    g2['levels'][-1] = [remap[t] for t in grammar['levels'][-1]]
    bottom_level_syms = set(grammar['levels'][-2])
    for sym in bottom_level_syms:
        g2['rules'][sym] = [tuple(remap[t] for t in rhs) for rhs in grammar['rules'][sym]]
    return g2


In [3]:

# --- Sanity check: generate a grammar, inspect structure and length distribution ---
set_seed(0)
_g_demo = generate_layered_cfg(depth=7, n_nt=3, n_t=3, rules_per_nt=3, seed=0, distinct_consecutive=True)
print("Level sizes:", [len(l) for l in _g_demo['levels']])
print("Sample rules for root:", _g_demo['rules'][_g_demo['root']])

_rng = random.Random(123)
_lengths, _term_counts = [], Counter()
for _ in range(3000):
    s = sample_string(_g_demo, _rng)
    _lengths.append(len(s))
    _term_counts.update(s)
_lengths.sort()
n = len(_lengths)
total = sum(_term_counts.values())
ent = -sum((v/total) * math.log2(v/total) for v in _term_counts.values())
print(f"Length pctiles 25/50/75/95: {_lengths[int(.25*n)]}/{_lengths[int(.5*n)]}/{_lengths[int(.75*n)]}/{_lengths[int(.95*n)]}")
print(f"Terminal distribution: {dict(_term_counts)}   unigram entropy: {ent:.3f} bits (max {math.log2(len(_g_demo['terminals'])):.3f})")

# Verify remap_terminals produces a structurally-identical grammar over a disjoint alphabet
_g_remap = remap_terminals(_g_demo, offset=100)
print("Remapped terminals:", _g_remap['terminals'])
assert _g_remap['rules'][_g_demo['root']] == _g_demo['rules'][_g_demo['root']], "NT-level rules should be untouched"


Level sizes: [1, 3, 3, 3, 3, 3, 3]
Sample rules for root: [(3, 1, 3), (2, 1, 2), (2, 1)]
Length pctiles 25/50/75/95: 169/205/235/275
Terminal distribution: {17: 223269, 18: 172480, 16: 210919}   unigram entropy: 1.577 bits (max 1.585)
Remapped terminals: {116, 117, 118}


## 2. Layered CYK-style Validity Parser

Standard CYK, specialized for our layered grammar (a symbol at level ℓ only ever
combines symbols at level ℓ+1, with rules of length 2 or 3). This is still a full
dynamic program over all substring splits — necessary for correctness, since a
span can in general be reached by more than one segmentation. Runtime is roughly
cubic in string length per level; empirically ~20-200ms per string at our target
lengths (150-400 tokens), which is fine for evaluation (hundreds to low thousands
of strings) but far too slow to call inside the training loop.

**A bug worth flagging explicitly** (caught during development, fixed here): a
naive implementation nests the length-3 rule check inside the length-2 rule
check for the same first two symbols. That's wrong — a length-3 rule
`(sym1, sym2, sym3) -> parent` doesn't require `(sym1, sym2)` to *also* be a
valid length-2 rule for any parent. The nested version silently drops valid
parses and produces false negatives on strings sampled directly from the
grammar. The version below checks both independently.


In [4]:

def build_reverse_rules(grammar):
    '''rev[level] : dict rhs_tuple -> list of parent symbols at that level.'''
    sym_level = {}
    for lvl, syms in enumerate(grammar['levels']):
        for s in syms:
            sym_level[s] = lvl
    rev = {lvl: {} for lvl in range(grammar['depth'] - 1)}
    for sym, rule_list in grammar['rules'].items():
        lvl = sym_level[sym]
        for rhs in rule_list:
            rev[lvl].setdefault(rhs, []).append(sym)
    return rev


def cyk_layered(string, grammar, rev_rules):
    '''Returns True iff `string` (a list of terminal ids) is generated by `grammar`.'''
    depth = grammar['depth']
    terminal_level = depth - 1
    n = len(string)
    if n == 0:
        return False

    table = {terminal_level: {}}
    for i in range(n):
        table[terminal_level][(i, i + 1)] = {string[i]}

    for lvl in range(terminal_level - 1, -1, -1):
        child_table = table[lvl + 1]
        cur = {}
        by_start = defaultdict(list)
        for (i, j), syms in child_table.items():
            by_start[i].append((j, syms))

        for i in range(n):
            if i not in by_start:
                continue
            for (j1, syms1) in by_start[i]:
                for sym1 in syms1:
                    if j1 not in by_start:
                        continue
                    for (j2, syms2) in by_start[j1]:
                        for sym2 in syms2:
                            rhs2 = (sym1, sym2)
                            parents2 = rev_rules[lvl].get(rhs2)
                            if parents2:
                                cur.setdefault((i, j2), set()).update(parents2)
                            # length-3 check is INDEPENDENT of the length-2 result above
                            if j2 in by_start:
                                for (j3, syms3) in by_start[j2]:
                                    for sym3 in syms3:
                                        rhs3 = (sym1, sym2, sym3)
                                        parents3 = rev_rules[lvl].get(rhs3)
                                        if parents3:
                                            cur.setdefault((i, j3), set()).update(parents3)
        table[lvl] = cur

    root_syms = table[0].get((0, n), set())
    return grammar['root'] in root_syms


In [5]:

# --- Sanity check: positive control (strings FROM the grammar must validate),
#     negative control (random strings must not), corruption sensitivity ---
_rev_demo = build_reverse_rules(_g_demo)
_rng = random.Random(1)

_n_ok, _n_total = 0, 30
_times = []
for _ in range(_n_total):
    s = sample_string(_g_demo, _rng)
    t0 = time.time()
    ok = cyk_layered(s, _g_demo, _rev_demo)
    _times.append(time.time() - t0)
    _n_ok += int(ok)
print(f"Positive control: {_n_ok}/{_n_total} grammar-sampled strings validated "
      f"(must be {_n_total}/{_n_total}) — avg {sum(_times)/len(_times):.3f}s, max {max(_times):.3f}s")
assert _n_ok == _n_total, "Parser bug: some grammar-sampled strings failed to validate!"

_terms = list(_g_demo['terminals'])
_n_neg_ok = 0
for _ in range(30):
    L = _rng.randint(150, 350)
    s = [_rng.choice(_terms) for _ in range(L)]
    _n_neg_ok += int(cyk_layered(s, _g_demo, _rev_demo))
print(f"Negative control: {_n_neg_ok}/30 random strings validated (should be ~0)")

_n_broken, _n_total2 = 0, 20
for _ in range(_n_total2):
    s = sample_string(_g_demo, _rng)
    pos = _rng.randint(0, len(s) - 1)
    s2 = list(s)
    choices = [t for t in _terms if t != s2[pos]]
    s2[pos] = _rng.choice(choices)
    _n_broken += int(not cyk_layered(s2, _g_demo, _rev_demo))
print(f"Single-token corruption: {_n_broken}/{_n_total2} correctly invalidated "
      f"(a few surviving is expected grammar ambiguity, not a bug)")


Positive control: 30/30 grammar-sampled strings validated (must be 30/30) — avg 0.335s, max 2.002s
Negative control: 0/30 random strings validated (should be ~0)
Single-token corruption: 14/20 correctly invalidated (a few surviving is expected grammar ambiguity, not a bug)


## 3. Diagnostics: Rule Overlap & N-gram Cross-Entropy Baseline

**Rule overlap** — Pre-check 4. Two independently-generated grammars should share
few rules by chance; high coincidental overlap would mean the "two languages" are
too similar to create genuine conflict.

**N-gram cross-entropy baseline** — this replaces a naive "bigram generation
accuracy" diagnostic, which we found to be misleading: with only 3 terminal
symbols, a purely greedy bigram continuation achieves ~93% *validity* on held-out
strings, purely because a 3-symbol alphabet leaves few ways to be locally
"wrong" — even though a bigram model captures none of the real long-range
structure. Cross-entropy under increasing n-gram order is a much sharper
diagnostic: if it plateaus at some small order, the task is locally solvable and
capacity won't genuinely bind on parsing; if it keeps falling well past order 2-3,
real long-range hierarchical dependencies remain that only deep attention can
capture. On our default grammar (depth 7, 3 NTs/level, 3 terminals, 3 rules/NT,
distinctness constraint on) cross-entropy fell from 1.55 (order 0) to 1.05 bits
(order 6) with **no plateau** — confirming genuine long-range structure.


In [6]:

def rule_overlap(g_a, g_b):
    '''Fraction of g_a's rules that coincide with g_b's rule for the
    position-matched symbol (matched by (level, index-within-level), since raw
    symbol ids differ between independently-generated grammars).'''
    def sym_to_pos(g):
        m = {}
        for lvl, syms in enumerate(g['levels']):
            for idx, s in enumerate(syms):
                m[s] = (lvl, idx)
        return m
    pos_a, pos_b = sym_to_pos(g_a), sym_to_pos(g_b)
    inv_b = {v: k for k, v in pos_b.items()}

    total, matching = 0, 0
    for sym_a, rules_a in g_a['rules'].items():
        lvl, idx = pos_a[sym_a]
        sym_b = inv_b[(lvl, idx)]
        rules_b = g_b['rules'][sym_b]
        rhs_a_pos = set(tuple(pos_a[s][1] for s in rhs) for rhs in rules_a)
        rhs_b_pos = set(tuple(pos_b[s][1] for s in rhs) for rhs in rules_b)
        total += len(rhs_a_pos)
        matching += len(rhs_a_pos & rhs_b_pos)
    return matching / total


def fit_ngram(grammar, order, n_samples, rng):
    counts = defaultdict(lambda: defaultdict(int))
    for _ in range(n_samples):
        s = sample_string(grammar, rng)
        ctx = tuple()
        for t in s:
            counts[ctx][t] += 1
            ctx = (ctx + (t,))[-order:] if order > 0 else tuple()
    return counts


def eval_ngram_crossentropy(grammar, counts, order, n_eval, rng, alpha=1.0):
    vocab = len(grammar['terminals'])
    total_nll, total_tok = 0.0, 0
    for _ in range(n_eval):
        s = sample_string(grammar, rng)
        ctx = tuple()
        for t in s:
            d = counts.get(ctx, {})
            tot = sum(d.values()) + alpha * vocab
            p = (d.get(t, 0) + alpha) / tot
            total_nll += -math.log2(p)
            total_tok += 1
            ctx = (ctx + (t,))[-order:] if order > 0 else tuple()
    return total_nll / total_tok


In [7]:

# --- Rule overlap between two independently-generated grammars ---
_g_a_demo = generate_layered_cfg(depth=7, n_nt=3, n_t=3, rules_per_nt=3, seed=10, distinct_consecutive=True)
_g_b_demo = generate_layered_cfg(depth=7, n_nt=3, n_t=3, rules_per_nt=3, seed=20, distinct_consecutive=True)
_overlap = rule_overlap(_g_a_demo, _g_b_demo)
print(f"Rule overlap fraction (seed 10 vs seed 20): {_overlap:.3f}  (pass threshold: < 0.25)")

# --- N-gram cross-entropy scaling: does it plateau early (bad) or keep falling (good)? ---
_rng_fit, _rng_eval = random.Random(100), random.Random(200)
print("\nN-gram cross-entropy on the default grammar (seed=0):")
for order in [0, 1, 2, 3, 4, 6]:
    counts = fit_ngram(_g_demo, order, 8000, _rng_fit)
    ce = eval_ngram_crossentropy(_g_demo, counts, order, 1000, _rng_eval)
    print(f"  order={order}: {ce:.4f} bits/token")
print("(Should keep decreasing well past order 2-3 with no plateau — confirms")
print(" real long-range structure a small transformer can't shortcut around.)")


Rule overlap fraction (seed 10 vs seed 20): 0.208  (pass threshold: < 0.25)

N-gram cross-entropy on the default grammar (seed=0):
  order=0: 1.5765 bits/token
  order=1: 1.2692 bits/token
  order=2: 1.1494 bits/token
  order=3: 1.1007 bits/token
  order=4: 1.0728 bits/token
  order=6: 1.0503 bits/token
(Should keep decreasing well past order 2-3 with no plateau — confirms
 real long-range structure a small transformer can't shortcut around.)


## 4. Model: GPT-2 with Rotary Position Embeddings

Following Allen-Zhu's finding that rotary/relative attention is essential for
learning these CFGs (their vanilla absolute-position GPT performed *worse* than
even a fixed-uniform-attention baseline on the harder grammars). Standard GPT-2
architecture (pre-norm, GELU MLP, tied input/output embeddings) with RoPE applied
to queries/keys instead of absolute position embeddings, and `d_model` as the
swept capacity knob. Fixed at 12 layers / 12 heads matching Allen-Zhu's own
`GPT_rot`; only `d_model` varies across the sweep.


In [8]:

def build_rope_cache(seq_len, dim, base=10000.0, device='cpu'):
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, device=device).float() / dim))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.einsum('i,j->ij', t, inv_freq)
    emb = torch.cat([freqs, freqs], dim=-1)
    return emb.cos(), emb.sin()


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(q, k, cos, sin):
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads, self.d_head = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        q, k = apply_rope(q, k, cos, sin)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)


class MLP(nn.Module):
    def __init__(self, d_model, mlp_ratio=4):
        super().__init__()
        self.fc1 = nn.Linear(d_model, mlp_ratio * d_model)
        self.fc2 = nn.Linear(mlp_ratio * d_model, d_model)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class Block(nn.Module):
    def __init__(self, d_model, n_heads, mlp_ratio=4):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model, mlp_ratio)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.ln1(x), cos, sin)
        x = x + self.mlp(self.ln2(x))
        return x


class GPTRope(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers=12, n_heads=12, max_seq_len=1024,
                 mlp_ratio=4, rope_base=10000.0):
        super().__init__()
        self.d_model, self.n_heads = d_model, n_heads
        self.d_head = d_model // n_heads
        self.max_seq_len, self.rope_base = max_seq_len, rope_base
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_heads, mlp_ratio) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        cos, sin = build_rope_cache(T, self.d_head, base=self.rope_base, device=idx.device)
        x = self.tok_emb(idx)
        for block in self.blocks:
            x = block(x, cos, sin)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    @torch.no_grad()
    def generate(self, prefix_tokens, max_new_tokens, temperature=1.0, eos_id=None):
        self.eval()
        idx = torch.tensor([prefix_tokens], dtype=torch.long, device=next(self.parameters()).device)
        for _ in range(max_new_tokens):
            with autocast_ctx():
                logits, _ = self(idx[:, -self.max_seq_len:])
            probs = torch.softmax(logits[0, -1].float() / temperature, dim=-1)
            nxt = torch.multinomial(probs, 1).item()
            idx = torch.cat([idx, torch.tensor([[nxt]], device=idx.device)], dim=1)
            if eos_id is not None and nxt == eos_id:
                break
        self.train()
        return idx[0].tolist()


In [9]:

# --- Sanity check: param counts across the full capacity sweep ---
CAPACITY_SWEEP = [48, 96, 192, 384, 768]  # d_model values; all divisible by N_HEADS=12
N_LAYERS, N_HEADS = 12, 12

for d_model in CAPACITY_SWEEP:
    m = GPTRope(vocab_size=10, d_model=d_model, n_layers=N_LAYERS, n_heads=N_HEADS, max_seq_len=512)
    print(f"d_model={d_model:4d}  d_head={d_model//N_HEADS:3d}  params={m.num_params():>11,d}")
    del m
print("\n(d_model=768 should land close to Allen-Zhu's reported 86M for GPT-2-small.)")


d_model=  48  d_head=  4  params=    339,840


d_model=  96  d_head=  8  params=  1,343,232
d_model= 192  d_head= 16  params=  5,340,672
d_model= 384  d_head= 32  params= 21,298,176
d_model= 768  d_head= 64  params= 85,063,680

(d_model=768 should land close to Allen-Zhu's reported 86M for GPT-2-small.)


## 5. Data Pipeline

Four conditions, following Allen-Zhu's own pre-training recipe (Appendix A.3):
concatenate `[BOS] sample [EOS]` blocks and cut into fixed-length windows —
a window may span the tail of one sentence and the head of the next.

| Condition    | Vocabulary                              | What it isolates |
|---|---|---|
| `mono_a`     | `{BOS_A, term_A..., EOS}`                | Language A's own ceiling |
| `mono_b`     | `{BOS_B, term_B..., EOS}`                | Language B's own ceiling |
| `bi_shared`  | `{BOS_A, BOS_B, terms(shared), EOS}`     | Joint training WITH embedding conflict |
| `bi_disjoint`| `{BOS_A, BOS_B, terms_A, terms_B, EOS}`  | Joint training WITHOUT embedding conflict (control) |

`bi_disjoint` uses `remap_terminals(grammar_b, offset)` so B's terminals occupy
entirely separate token ids from A's — same two grammars, same computational
load, zero shared-embedding conflict. The gap between `bi_shared` and
`bi_disjoint` on the same language is our ε signal.


In [10]:

@dataclass
class VocabInfo:
    vocab_size: int
    bos_a: int
    bos_b: int
    eos: int
    terms_a: set
    terms_b: set  # for mono conditions, terms_b is unused / empty


def build_vocab(g_a, g_b, condition):
    '''Assign token ids for BOS_A, BOS_B, EOS, and terminal symbols, per condition.
    Terminal ids for g_a and (remapped, if needed) g_b are taken as-is from the
    grammar dicts, which is fine since generate_layered_cfg / remap_terminals
    already assign small contiguous integer ids starting at 0.'''
    max_term = max(max(g_a['terminals']), max(g_b['terminals']))
    bos_a = max_term + 1
    bos_b = max_term + 2
    eos = max_term + 3
    vocab_size = eos + 1
    return VocabInfo(vocab_size=vocab_size, bos_a=bos_a, bos_b=bos_b, eos=eos,
                      terms_a=set(g_a['terminals']), terms_b=set(g_b['terminals']))


def make_batch(condition, g_a, g_b, vocab, batch_size, window, rng):
    '''Build one training batch: concatenate BOS/EOS-wrapped samples until we
    have enough tokens, then reshape into (batch_size, window+1) and split into
    (x, y) for next-token prediction.

    condition in {'mono_a', 'mono_b', 'bi_shared', 'bi_disjoint'}.
    For bi_* conditions, g_b passed in should ALREADY be the appropriately-terminal
    grammar for the condition (i.e. remapped for bi_disjoint, not remapped/original
    ids for bi_shared) -- see build_language_pair() below which prepares both.
    '''
    buf = []
    target_len = batch_size * (window + 1)
    while len(buf) < target_len:
        if condition == 'mono_a':
            s = sample_string(g_a, rng); buf.append(vocab.bos_a); buf.extend(s); buf.append(vocab.eos)
        elif condition == 'mono_b':
            s = sample_string(g_b, rng); buf.append(vocab.bos_b); buf.extend(s); buf.append(vocab.eos)
        else:  # bi_shared or bi_disjoint: 50/50 mixture
            if rng.random() < 0.5:
                s = sample_string(g_a, rng); buf.append(vocab.bos_a); buf.extend(s); buf.append(vocab.eos)
            else:
                s = sample_string(g_b, rng); buf.append(vocab.bos_b); buf.extend(s); buf.append(vocab.eos)
    buf = buf[:target_len]
    t = torch.tensor(buf, dtype=torch.long).view(batch_size, window + 1).contiguous()
    return t[:, :-1].contiguous(), t[:, 1:].contiguous()


def build_language_pair(seed_a=10, seed_b=20, depth=7, n_nt=3, n_t=3, rules_per_nt=3):
    '''Build G_A and G_B once, plus the terminal-remapped copy of G_B used for
    the bi_disjoint condition. Returns (g_a, g_b_shared_ids, g_b_disjoint_ids).'''
    g_a = generate_layered_cfg(depth=depth, n_nt=n_nt, n_t=n_t, rules_per_nt=rules_per_nt,
                                seed=seed_a, distinct_consecutive=True)
    g_b = generate_layered_cfg(depth=depth, n_nt=n_nt, n_t=n_t, rules_per_nt=rules_per_nt,
                                seed=seed_b, distinct_consecutive=True)
    offset = max(g_a['terminals']) + 1
    g_b_disjoint = remap_terminals(g_b, offset=offset)
    return g_a, g_b, g_b_disjoint


In [11]:

# --- Sanity check: build a language pair, inspect vocab construction per condition ---
_ga, _gb, _gb_disjoint = build_language_pair()
print("G_A terminals:", _ga['terminals'])
print("G_B terminals (shared-alphabet id space):", _gb['terminals'])
print("G_B terminals (disjoint, remapped):", _gb_disjoint['terminals'])

_vocab_shared = build_vocab(_ga, _gb, 'bi_shared')
_vocab_disjoint = build_vocab(_ga, _gb_disjoint, 'bi_disjoint')
print(f"\nbi_shared vocab_size={_vocab_shared.vocab_size}  (expect small: shared terms + 2 BOS + EOS)")
print(f"bi_disjoint vocab_size={_vocab_disjoint.vocab_size}  (expect larger: disjoint terms + 2 BOS + EOS)")

_rng = random.Random(0)
x, y = make_batch('bi_shared', _ga, _gb, _vocab_shared, batch_size=4, window=32, rng=_rng)
print("\nSample bi_shared batch shapes:", x.shape, y.shape)
print("x[0]:", x[0].tolist())


G_A terminals: {16, 17, 18}
G_B terminals (shared-alphabet id space): {16, 17, 18}
G_B terminals (disjoint, remapped): {35, 36, 37}

bi_shared vocab_size=22  (expect small: shared terms + 2 BOS + EOS)
bi_disjoint vocab_size=41  (expect larger: disjoint terms + 2 BOS + EOS)

Sample bi_shared batch shapes: torch.Size([4, 32]) torch.Size([4, 32])
x[0]: [20, 16, 17, 18, 16, 16, 17, 18, 17, 16, 17, 18, 16, 17, 16, 18, 16, 17, 16, 17, 18, 17, 18, 16, 16, 18, 17, 16, 17, 18, 17, 17]


## 6. Training Loop

Hyperparameters follow Allen-Zhu Appendix A.3 exactly (β=(0.9,0.98), weight decay
0.1, lr 3e-4, linear decay, no warmup — they explicitly note warmup isn't needed
for CFG pretraining). Batch size and step count are configurable; defaults match
the paper's own single-grammar recipe (batch 96, window 512, 100K steps ≈ 4.9B
tokens), doubled to 200K for bilingual conditions as a starting point — extend
further if per-language loss curves haven't plateaued (see convergence check
below; **never trust a fixed-step accuracy number over a converged one** — this
is the exact trap that made Chunk A of the prior TinyGPT experiment
inconclusive).

**bf16 autocast (`USE_BF16`, set in Section 0).** The forward pass runs under
`torch.autocast(dtype=torch.bfloat16)` on CUDA, a deliberate deviation from
Allen-Zhu's plain-float32 recipe, purely for throughput on modern tensor-core
GPUs (A100/H100/H200). No gradient scaler is needed — bf16 keeps fp32's exponent
range, unlike fp16, so this is a low-risk change at this model scale. Applied
consistently everywhere a forward pass happens (training, `eval_crossentropy`,
`generate`, `per_position_loss`) so the Bi-shared vs. Bi-disjoint comparison that
ε is built from stays apples-to-apples. **Recommended one-time sanity check**
before trusting a full 100-200K-step run to it: run Pre-check 1 for a few
thousand steps with `USE_BF16 = True`, then again with `USE_BF16 = False`, and
confirm the loss curves track closely. Set `USE_BF16 = False` globally in
Section 0 to fall back to Allen-Zhu's exact float32 setup at any time.


In [12]:

def train_model(condition, g_a, g_b, vocab, d_model, n_steps, batch_size=96, window=512,
                 n_layers=12, n_heads=12, lr=3e-4, seed=0, log_every=1000, eval_every=5000,
                 eval_fn=None, ckpt_path=None, resume_from=None):
    '''
    Trains a GPTRope model on `condition`'s data mixture. Returns (model, history)
    where history is a dict of step -> {'loss': float, 'eval': eval_fn(model) result or None}.

    eval_fn, if given, is called periodically as eval_fn(model, step) and its
    return value (expected to be a small dict of scalars, e.g. per-language loss)
    is stored in the history for convergence tracking -- pass something cheap,
    NOT full generation-accuracy (that's for the post-hoc evaluation stage).
    '''
    set_seed(seed)
    rng = random.Random(seed)
    model = GPTRope(vocab_size=vocab.vocab_size, d_model=d_model, n_layers=n_layers,
                     n_heads=n_heads, max_seq_len=window).to(DEVICE)
    if resume_from is not None:
        model.load_state_dict(torch.load(resume_from, map_location=DEVICE))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.98), weight_decay=0.1)
    sched = torch.optim.lr_scheduler.LinearLR(opt, start_factor=1.0, end_factor=0.0, total_iters=n_steps)

    history = {}
    t0 = time.time()
    for step in range(1, n_steps + 1):
        x, y = make_batch(condition, g_a, g_b, vocab, batch_size, window, rng)
        x, y = x.to(DEVICE), y.to(DEVICE)
        with autocast_ctx():
            logits, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()

        if step % log_every == 0 or step == 1:
            elapsed = time.time() - t0
            entry = {'loss': loss.item(), 'elapsed_s': elapsed}
            if eval_fn is not None and step % eval_every == 0:
                entry['eval'] = eval_fn(model, step)
            history[step] = entry
            eval_str = f"  eval={entry.get('eval')}" if 'eval' in entry else ""
            print(f"[{condition} d_model={d_model} seed={seed}] step {step}/{n_steps}  "
                  f"loss={loss.item():.4f}  ({elapsed:.0f}s){eval_str}")

        if ckpt_path is not None and (step % (eval_every * 2) == 0 or step == n_steps):
            torch.save(model.state_dict(), ckpt_path)

    return model, history


def check_convergence(history, window_frac=0.1, tol=0.01):
    '''Simple convergence check: compare the mean loss over the last `window_frac`
    of logged steps against the mean loss over the preceding equal-sized window.
    Returns (converged: bool, relative_change: float).'''
    steps = sorted(history.keys())
    n = len(steps)
    w = max(1, int(n * window_frac))
    if n < 2 * w:
        return False, float('inf')
    recent = [history[s]['loss'] for s in steps[-w:]]
    prior = [history[s]['loss'] for s in steps[-2*w:-w]]
    recent_mean, prior_mean = sum(recent) / len(recent), sum(prior) / len(prior)
    rel_change = abs(recent_mean - prior_mean) / max(abs(prior_mean), 1e-8)
    return rel_change < tol, rel_change


## 7. Evaluation

Two metrics, following Allen-Zhu's own Results 1-3:

1. **Generation accuracy** — sample from the model (temperature=1, multinomial,
   matching the paper), check validity with the CYK parser. Test both `c=0`
   (generate from BOS alone) and `c=50` (complete a 50-token real prefix).
2. **Cross-entropy loss** — standard teacher-forced next-token loss on fresh
   held-out samples. This is the metric that actually drives the ε computation
   below: **ε_measured is a relative comparison between Bi-shared and
   Bi-disjoint on the same eval data, so it does not require an absolute
   ground-truth entropy floor** — we sidestep the (much harder) problem of
   computing the CFG's exact prefix-probability distribution.


In [13]:

@torch.no_grad()
def eval_generation_accuracy(model, grammar, rev_rules, bos_id, eos_id, n_samples=200,
                              cut=0, max_new_tokens=450, temperature=1.0, rng=None, ref_grammar_for_prefix=None):
    '''
    cut=0  : generate from BOS alone.
    cut=k  : take a real k-token prefix from `ref_grammar_for_prefix` (or `grammar`
             if not given), feed [BOS, prefix...], generate the rest, check validity
             of the FULL resulting string (prefix + completion).
    '''
    if rng is None:
        rng = random.Random(0)
    ref_grammar = ref_grammar_for_prefix or grammar
    n_valid = 0
    lengths = []
    for _ in range(n_samples):
        if cut == 0:
            prefix_tokens = [bos_id]
            true_prefix_terms = []
        else:
            full = sample_string(ref_grammar, rng)
            if len(full) <= cut:
                continue
            true_prefix_terms = full[:cut]
            prefix_tokens = [bos_id] + true_prefix_terms
        gen = model.generate(prefix_tokens, max_new_tokens=max_new_tokens,
                              temperature=temperature, eos_id=eos_id)
        body = gen[1:]  # drop BOS
        if eos_id in body:
            body = body[:body.index(eos_id)]
        body = [t for t in body if t in grammar['terminals']]
        lengths.append(len(body))
        ok = len(body) > 0 and cyk_layered(body, grammar, rev_rules)
        n_valid += int(ok)
    return {'accuracy': n_valid / max(1, n_samples), 'n': n_samples,
            'mean_len': sum(lengths) / max(1, len(lengths))}


@torch.no_grad()
def eval_crossentropy(model, condition_for_eval, g_a, g_b, vocab, n_batches=20,
                       batch_size=32, window=512, seed=12345):
    '''Per-language held-out cross-entropy. condition_for_eval selects which
    single language's data to evaluate on ('mono_a' or 'mono_b' as data sources
    -- works regardless of what condition the model was TRAINED under, since we
    just want the model's loss on fresh language-A-only or language-B-only data
    with the correct BOS tag).'''
    rng = random.Random(seed)
    total_loss, total_count = 0.0, 0
    for _ in range(n_batches):
        x, y = make_batch(condition_for_eval, g_a, g_b, vocab, batch_size, window, rng)
        x, y = x.to(DEVICE), y.to(DEVICE)
        with autocast_ctx():
            _, loss = model(x, y)
        total_loss += loss.item() * x.numel()
        total_count += x.numel()
    return total_loss / total_count


def compute_epsilon(loss_bi_shared_a, loss_bi_disjoint_a, loss_bi_shared_b, loss_bi_disjoint_b):
    '''ε_measured = average, over languages, of (Bi-shared loss - Bi-disjoint loss).
    Positive ε means shared terminals cost the model accuracy relative to the
    no-conflict control -- this is the core signal the whole experiment is
    built to detect.'''
    eps_a = loss_bi_shared_a - loss_bi_disjoint_a
    eps_b = loss_bi_shared_b - loss_bi_disjoint_b
    return {'eps_a': eps_a, 'eps_b': eps_b, 'eps_mean': (eps_a + eps_b) / 2}


## 8. Pre-checks — RUN THESE BEFORE THE MAIN SWEEP

Each gates the next. Do not proceed to the main sweep (Section 9) until all five
pass, or you'll spend real GPU-days rediscovering problems we can catch cheaply
first. Every threshold below was chosen based on the CPU-scale diagnostics run
while designing this notebook (see Sections 1-3) — Pre-checks 1-3 are the ones
that need real training time; run them at reduced step counts first if you want
a fast signal before committing to the full 100-200K steps.


In [14]:

# ---- Pre-check config ----
PRECHECK_STEPS = 100_000      # bump if convergence check fails
PRECHECK_BATCH = 96
PRECHECK_WINDOW = 512

ga_pc, gb_pc, gb_disjoint_pc = build_language_pair(seed_a=10, seed_b=20,
                                                    depth=7, n_nt=3, n_t=3, rules_per_nt=3)
rev_a_pc = build_reverse_rules(ga_pc)
rev_b_pc = build_reverse_rules(gb_pc)


In [ ]:

# ---- Pre-check 1: monolingual learnability (d_model=768, one seed) ----
# PASS criterion: >=95% generation accuracy at both cut=0 and cut=50.
vocab_mono_a = build_vocab(ga_pc, gb_pc, 'mono_a')  # gb_pc unused for mono_a but build_vocab needs both grammars

def cheap_eval_fn_mono_a(model, step):
    return {'ce': eval_crossentropy(model, 'mono_a', ga_pc, gb_pc, vocab_mono_a, n_batches=5)}

model_mono_a, hist_mono_a = train_model(
    'mono_a', ga_pc, gb_pc, vocab_mono_a, d_model=768, n_steps=PRECHECK_STEPS,
    batch_size=PRECHECK_BATCH, window=PRECHECK_WINDOW, seed=0,
    eval_fn=cheap_eval_fn_mono_a, eval_every=10_000,
    ckpt_path=os.path.join(RESULTS_DIR, "checkpoints", "precheck1_mono_a.pt"),
)
converged, rel_change = check_convergence(hist_mono_a)
print(f"\nConverged: {converged} (relative change {rel_change:.4f}) -- extend PRECHECK_STEPS if False")

acc0 = eval_generation_accuracy(model_mono_a, ga_pc, rev_a_pc, vocab_mono_a.bos_a, vocab_mono_a.eos,
                                 n_samples=200, cut=0)
acc50 = eval_generation_accuracy(model_mono_a, ga_pc, rev_a_pc, vocab_mono_a.bos_a, vocab_mono_a.eos,
                                  n_samples=200, cut=50, ref_grammar_for_prefix=ga_pc)
print(f"Mono-A generation accuracy: cut=0: {acc0['accuracy']:.3f}   cut=50: {acc50['accuracy']:.3f}")
print("PASS" if (acc0['accuracy'] >= 0.95 and acc50['accuracy'] >= 0.95) else
      "FAIL -- grammar too hard at rules_per_nt=3. Try rules_per_nt=2, or check convergence first.")


[mono_a d_model=768 seed=0] step 1/100000  loss=3.2031  (2s)
[mono_a d_model=768 seed=0] step 1000/100000  loss=0.7693  (351s)
[mono_a d_model=768 seed=0] step 2000/100000  loss=0.7589  (719s)
[mono_a d_model=768 seed=0] step 3000/100000  loss=0.7555  (1084s)
[mono_a d_model=768 seed=0] step 4000/100000  loss=0.7517  (1452s)
[mono_a d_model=768 seed=0] step 5000/100000  loss=0.7502  (1817s)
[mono_a d_model=768 seed=0] step 6000/100000  loss=0.7488  (2197s)
[mono_a d_model=768 seed=0] step 7000/100000  loss=0.7518  (2600s)
[mono_a d_model=768 seed=0] step 8000/100000  loss=0.7496  (2987s)
[mono_a d_model=768 seed=0] step 9000/100000  loss=0.7471  (3381s)
[mono_a d_model=768 seed=0] step 10000/100000  loss=0.7458  (3781s)  eval={'ce': 0.7482597708702088}
[mono_a d_model=768 seed=0] step 11000/100000  loss=0.7491  (4162s)
[mono_a d_model=768 seed=0] step 12000/100000  loss=0.7470  (4557s)
[mono_a d_model=768 seed=0] step 13000/100000  loss=0.7449  (4951s)
[mono_a d_model=768 seed=0] step 

In [ ]:

# ---- Pre-check 1 (cont'd): repeat for Mono-B, then Pre-check 2 (difficulty matching) ----
vocab_mono_b = build_vocab(ga_pc, gb_pc, 'mono_b')

def cheap_eval_fn_mono_b(model, step):
    return {'ce': eval_crossentropy(model, 'mono_b', ga_pc, gb_pc, vocab_mono_b, n_batches=5)}

model_mono_b, hist_mono_b = train_model(
    'mono_b', ga_pc, gb_pc, vocab_mono_b, d_model=768, n_steps=PRECHECK_STEPS,
    batch_size=PRECHECK_BATCH, window=PRECHECK_WINDOW, seed=0,
    eval_fn=cheap_eval_fn_mono_b, eval_every=10_000,
    ckpt_path=os.path.join(RESULTS_DIR, "checkpoints", "precheck1_mono_b.pt"),
)
converged_b, rel_change_b = check_convergence(hist_mono_b)
print(f"\nConverged: {converged_b} (relative change {rel_change_b:.4f})")

acc0_b = eval_generation_accuracy(model_mono_b, gb_pc, rev_b_pc, vocab_mono_b.bos_b, vocab_mono_b.eos,
                                   n_samples=200, cut=0)
acc50_b = eval_generation_accuracy(model_mono_b, gb_pc, rev_b_pc, vocab_mono_b.bos_b, vocab_mono_b.eos,
                                    n_samples=200, cut=50, ref_grammar_for_prefix=gb_pc)
print(f"Mono-B generation accuracy: cut=0: {acc0_b['accuracy']:.3f}   cut=50: {acc50_b['accuracy']:.3f}")

# ---- Pre-check 2: difficulty matching ----
gap0 = abs(acc0['accuracy'] - acc0_b['accuracy'])
gap50 = abs(acc50['accuracy'] - acc50_b['accuracy'])
print(f"\nPre-check 2 -- Mono-A vs Mono-B gap: cut=0: {gap0:.3f}   cut=50: {gap50:.3f}")
print("PASS" if (gap0 <= 0.03 and gap50 <= 0.03) else
      "FAIL -- regenerate the harder grammar (bump its seed) and rerun Pre-checks 1-2.")


In [ ]:

# ---- Pre-check 3: capacity binding (Mono-A across the full d_model sweep) ----
# PASS criterion: clear transition from near-chance (small d_model) to near-ceiling
# (d_model=768), with a >20-point gap between the smallest and largest d_model.
# Uses the SAME step budget as Pre-check 1 at each capacity -- if a small-d_model
# run looks stuck at chance, check its OWN convergence curve before concluding
# "capacity-limited" vs "just needs more steps" (the Chunk A lesson).

capacity_binding_results = {}
for d_model in CAPACITY_SWEEP:
    if d_model == max(CAPACITY_SWEEP):
        # reuse Pre-check 1's model/result instead of retraining
        capacity_binding_results[d_model] = {'acc0': acc0['accuracy'], 'acc50': acc50['accuracy'],
                                              'converged': converged, 'rel_change': rel_change}
        continue
    m, h = train_model('mono_a', ga_pc, gb_pc, vocab_mono_a, d_model=d_model, n_steps=PRECHECK_STEPS,
                        batch_size=PRECHECK_BATCH, window=PRECHECK_WINDOW, seed=0,
                        eval_fn=cheap_eval_fn_mono_a, eval_every=10_000)
    conv, rel = check_convergence(h)
    a0 = eval_generation_accuracy(m, ga_pc, rev_a_pc, vocab_mono_a.bos_a, vocab_mono_a.eos, n_samples=150, cut=0)
    a50 = eval_generation_accuracy(m, ga_pc, rev_a_pc, vocab_mono_a.bos_a, vocab_mono_a.eos, n_samples=150,
                                    cut=50, ref_grammar_for_prefix=ga_pc)
    capacity_binding_results[d_model] = {'acc0': a0['accuracy'], 'acc50': a50['accuracy'],
                                          'converged': conv, 'rel_change': rel}
    print(f"d_model={d_model}: acc0={a0['accuracy']:.3f} acc50={a50['accuracy']:.3f} "
          f"converged={conv} (rel_change={rel:.4f})")
    del m

print("\nSummary:")
for d_model, r in capacity_binding_results.items():
    print(f"  d_model={d_model:4d}: acc0={r['acc0']:.3f} acc50={r['acc50']:.3f} converged={r['converged']}")
gap = capacity_binding_results[max(CAPACITY_SWEEP)]['acc50'] - capacity_binding_results[min(CAPACITY_SWEEP)]['acc50']
print(f"\nGap (largest - smallest d_model), cut=50: {gap:.3f}")
print("PASS" if gap >= 0.20 else
      "FAIL -- if ceiling everywhere: bump rules_per_nt to 4. If chance everywhere "
      "AND unconverged: extend PRECHECK_STEPS before concluding anything.")


In [ ]:

# ---- Pre-check 4: rule overlap audit (cheap, no training needed) ----
overlap_pc = rule_overlap(ga_pc, gb_pc)
print(f"Rule overlap (G_A vs G_B, seeds 10 & 20): {overlap_pc:.3f}")
print("PASS" if overlap_pc < 0.25 else "FAIL -- regenerate G_B with a different seed.")


In [ ]:

# ---- Pre-check 5: n-gram cross-entropy scaling (cheap, no GPU needed) ----
# PASS criterion: cross-entropy keeps falling well past order 2-3 with no plateau
# (confirms genuine long-range structure, not a local-pattern-matching shortcut).
print("N-gram cross-entropy scaling for G_A:")
_rng_fit, _rng_eval = random.Random(300), random.Random(301)
ce_by_order = {}
for order in [0, 1, 2, 3, 4, 6]:
    counts = fit_ngram(ga_pc, order, 8000, _rng_fit)
    ce = eval_ngram_crossentropy(ga_pc, counts, order, 1000, _rng_eval)
    ce_by_order[order] = ce
    print(f"  order={order}: {ce:.4f} bits/token")

drop_2_to_6 = ce_by_order[2] - ce_by_order[6]
print(f"\nDrop from order=2 to order=6: {drop_2_to_6:.4f} bits")
print("PASS" if drop_2_to_6 > 0.05 else
      "FAIL -- grammar is locally predictable; regenerate with more rules/NT or check distinctness constraint.")


## 9. Main Sweep

Only run this after all five pre-checks pass. Full grid: 4 conditions × 5
capacities × 3 seeds = 60 training runs. This is the expensive part — budget
accordingly (see the compute estimate in the design doc: roughly 100-150
GPU-hours on a single A100 for the full grid at 100-200K steps each, dominated
by the `d_model=768` runs).

Each run's model is discarded after evaluation (only history + eval metrics are
kept) to avoid holding 60 checkpoints in memory; adjust `SAVE_ALL_CHECKPOINTS`
if you want them retained for later diagnostics (Section 10 needs the
`bi_shared` checkpoints specifically for the tag-swap test).


In [ ]:

SWEEP_STEPS = 200_000       # bilingual conditions see each language half as often -> double budget
SWEEP_SEEDS = [0, 1, 2]
SAVE_ALL_CHECKPOINTS = True  # set False to save disk space; Section 10 diagnostics need bi_shared checkpoints

# Build the (possibly re-generated, if pre-checks required changes) grammar pair.
# If you changed rules_per_nt/seeds while iterating on pre-checks above, rebuild here explicitly:
GA, GB, GB_DISJOINT = build_language_pair(seed_a=10, seed_b=20, depth=7, n_nt=3, n_t=3, rules_per_nt=3)
REV_A = build_reverse_rules(GA)
REV_B = build_reverse_rules(GB)  # same structural grammar as GB_DISJOINT, just relabeled terminals -- rules identical
REV_B_DISJOINT = build_reverse_rules(GB_DISJOINT)

VOCAB_MONO_A = build_vocab(GA, GB, 'mono_a')
VOCAB_MONO_B = build_vocab(GA, GB, 'mono_b')
VOCAB_BI_SHARED = build_vocab(GA, GB, 'bi_shared')
VOCAB_BI_DISJOINT = build_vocab(GA, GB_DISJOINT, 'bi_disjoint')

sweep_results = []  # list of dicts, one per (condition, d_model, seed) run


In [ ]:

def run_one_sweep_point(condition, d_model, seed):
    if condition == 'mono_a':
        g_a, g_b, vocab = GA, GB, VOCAB_MONO_A
    elif condition == 'mono_b':
        g_a, g_b, vocab = GA, GB, VOCAB_MONO_B
    elif condition == 'bi_shared':
        g_a, g_b, vocab = GA, GB, VOCAB_BI_SHARED
    elif condition == 'bi_disjoint':
        g_a, g_b, vocab = GA, GB_DISJOINT, VOCAB_BI_DISJOINT
    else:
        raise ValueError(condition)

    def cheap_eval(model, step):
        out = {}
        if condition in ('mono_a', 'bi_shared', 'bi_disjoint'):
            src_cond = 'mono_a' if condition != 'bi_disjoint' else 'mono_a'
            out['ce_a'] = eval_crossentropy(model, 'mono_a', g_a, g_b, vocab, n_batches=5)
        if condition in ('mono_b', 'bi_shared', 'bi_disjoint'):
            out['ce_b'] = eval_crossentropy(model, 'mono_b', g_a, g_b, vocab, n_batches=5)
        return out

    ckpt_path = None
    if SAVE_ALL_CHECKPOINTS:
        ckpt_path = os.path.join(RESULTS_DIR, "checkpoints", f"{condition}_d{d_model}_s{seed}.pt")

    model, history = train_model(condition, g_a, g_b, vocab, d_model=d_model, n_steps=SWEEP_STEPS,
                                  batch_size=96, window=512, seed=seed,
                                  eval_fn=cheap_eval, eval_every=10_000, ckpt_path=ckpt_path)
    converged, rel_change = check_convergence(history)

    result = {'condition': condition, 'd_model': d_model, 'seed': seed,
              'converged': converged, 'rel_change': rel_change}

    # Final cross-entropy on fresh held-out data (this feeds directly into epsilon)
    if condition != 'mono_b':
        result['ce_a'] = eval_crossentropy(model, 'mono_a', g_a, g_b, vocab, n_batches=40)
    if condition != 'mono_a':
        result['ce_b'] = eval_crossentropy(model, 'mono_b', g_a, g_b, vocab, n_batches=40)

    # Generation accuracy (secondary metric, matching Allen-Zhu's own primary metric)
    if condition != 'mono_b':
        rev = REV_A
        a0 = eval_generation_accuracy(model, g_a, rev, vocab.bos_a, vocab.eos, n_samples=150, cut=0)
        a50 = eval_generation_accuracy(model, g_a, rev, vocab.bos_a, vocab.eos, n_samples=150,
                                        cut=50, ref_grammar_for_prefix=g_a)
        result['acc_a_cut0'], result['acc_a_cut50'] = a0['accuracy'], a50['accuracy']
    if condition != 'mono_a':
        g_b_eval = g_b
        rev = REV_B_DISJOINT if condition == 'bi_disjoint' else REV_B
        b0 = eval_generation_accuracy(model, g_b_eval, rev, vocab.bos_b, vocab.eos, n_samples=150, cut=0)
        b50 = eval_generation_accuracy(model, g_b_eval, rev, vocab.bos_b, vocab.eos, n_samples=150,
                                        cut=50, ref_grammar_for_prefix=g_b_eval)
        result['acc_b_cut0'], result['acc_b_cut50'] = b0['accuracy'], b50['accuracy']

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


In [ ]:

# Full sweep loop. This is the expensive cell -- consider running it condition-by-condition
# or capacity-by-capacity across separate sessions/checkpointed cells if working interactively.
CONDITIONS = ['mono_a', 'mono_b', 'bi_shared', 'bi_disjoint']

for condition in CONDITIONS:
    for d_model in CAPACITY_SWEEP:
        for seed in SWEEP_SEEDS:
            print(f"\n{'='*70}\nRunning: condition={condition} d_model={d_model} seed={seed}\n{'='*70}")
            result = run_one_sweep_point(condition, d_model, seed)
            sweep_results.append(result)
            # incremental save so a crash doesn't lose completed runs
            with open(os.path.join(RESULTS_DIR, "sweep_results.json"), "w") as f:
                json.dump(sweep_results, f, indent=2)

print(f"\nSweep complete: {len(sweep_results)} runs.")


## 10. ε Computation, N* Estimation, and Diagnostics

In [ ]:

import statistics

def aggregate_sweep(sweep_results):
    '''Group by (condition, d_model), average cross-entropy and accuracy over seeds.'''
    grouped = defaultdict(list)
    for r in sweep_results:
        grouped[(r['condition'], r['d_model'])].append(r)
    agg = {}
    for key, runs in grouped.items():
        entry = {'n_seeds': len(runs)}
        for metric in ['ce_a', 'ce_b', 'acc_a_cut0', 'acc_a_cut50', 'acc_b_cut0', 'acc_b_cut50']:
            vals = [r[metric] for r in runs if metric in r]
            if vals:
                entry[metric + '_mean'] = statistics.mean(vals)
                entry[metric + '_std'] = statistics.stdev(vals) if len(vals) > 1 else 0.0
        agg[key] = entry
    return agg


def compute_epsilon_curve(agg, capacities):
    '''epsilon(d_model) = mean over languages of (Bi-shared CE - Bi-disjoint CE).'''
    eps_curve = {}
    for d_model in capacities:
        bs = agg.get(('bi_shared', d_model), {})
        bd = agg.get(('bi_disjoint', d_model), {})
        if 'ce_a_mean' in bs and 'ce_a_mean' in bd and 'ce_b_mean' in bs and 'ce_b_mean' in bd:
            eps_a = bs['ce_a_mean'] - bd['ce_a_mean']
            eps_b = bs['ce_b_mean'] - bd['ce_b_mean']
            eps_curve[d_model] = {'eps_a': eps_a, 'eps_b': eps_b, 'eps_mean': (eps_a + eps_b) / 2}
    return eps_curve


def estimate_n_star(eps_curve, capacities, threshold=0.05):
    '''Smallest d_model where eps_mean drops below `threshold` nats. Returns None
    if no capacity in the sweep reaches threshold (extend CAPACITY_SWEEP upward).'''
    for d_model in sorted(capacities):
        if d_model in eps_curve and eps_curve[d_model]['eps_mean'] < threshold:
            return d_model
    return None


In [ ]:

# Load results (works whether this cell runs right after the sweep, or in a fresh
# session against a previously-saved sweep_results.json)
with open(os.path.join(RESULTS_DIR, "sweep_results.json")) as f:
    sweep_results = json.load(f)

agg = aggregate_sweep(sweep_results)
eps_curve = compute_epsilon_curve(agg, CAPACITY_SWEEP)
n_star = estimate_n_star(eps_curve, CAPACITY_SWEEP)

print("epsilon(d_model):")
for d_model in CAPACITY_SWEEP:
    if d_model in eps_curve:
        e = eps_curve[d_model]
        print(f"  d_model={d_model:4d}: eps_a={e['eps_a']:+.4f}  eps_b={e['eps_b']:+.4f}  eps_mean={e['eps_mean']:+.4f}")
print(f"\nEstimated N* (smallest d_model with eps_mean < 0.05 nats): {n_star}")


In [ ]:

# --- Plot: per-language cross-entropy across capacity, Bi-shared vs Bi-disjoint vs Mono ---
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, lang, ce_key in [(axes[0], 'A', 'ce_a'), (axes[1], 'B', 'ce_b')]:
    for condition, style in [('mono_a' if lang == 'A' else 'mono_b', 'k--'),
                              ('bi_shared', 'r-o'), ('bi_disjoint', 'b-s')]:
        xs, ys, yerrs = [], [], []
        for d_model in CAPACITY_SWEEP:
            entry = agg.get((condition, d_model), {})
            if ce_key + '_mean' in entry:
                xs.append(d_model); ys.append(entry[ce_key + '_mean']); yerrs.append(entry[ce_key + '_std'])
        if xs:
            ax.errorbar(xs, ys, yerr=yerrs, fmt=style, label=condition, capsize=3)
    ax.set_xscale('log', base=2)
    ax.set_xlabel('d_model')
    ax.set_ylabel(f'Cross-entropy, Language {lang} (nats/token)')
    ax.set_title(f'Language {lang}')
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "ce_by_capacity.png"), dpi=150)
plt.show()

# --- Plot: epsilon(d_model) ---
fig, ax = plt.subplots(figsize=(6, 4.5))
xs = [d for d in CAPACITY_SWEEP if d in eps_curve]
ys = [eps_curve[d]['eps_mean'] for d in xs]
ax.plot(xs, ys, 'g-o')
ax.axhline(0.05, color='gray', linestyle=':', label='N* threshold (0.05 nats)')
ax.set_xscale('log', base=2)
ax.set_xlabel('d_model')
ax.set_ylabel('epsilon_mean (Bi-shared CE - Bi-disjoint CE, nats/token)')
ax.set_title('Measured epsilon vs capacity')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "epsilon_curve.png"), dpi=150)
plt.show()


### Diagnostic 1: Tag-swap test

If the model has learned genuine grammar-conditional parsing (not just ignoring
the BOS tag), feeding G_A sentences with the wrong BOS tag (`BOS_B`) should
collapse validity toward chance. Run this on the `bi_shared` checkpoints across
the capacity sweep to see whether — and at what capacity — the model actually
uses the tag to switch parsing strategy.


In [ ]:

@torch.no_grad()
def tag_swap_test(model, g_a, rev_a, vocab, n_samples=100, cut=50, rng=None):
    '''Feed G_A sentences with the WRONG (B) BOS tag; a model doing genuine
    grammar-conditional parsing should show a large accuracy drop vs the
    correct-tag baseline.'''
    if rng is None:
        rng = random.Random(0)
    correct = eval_generation_accuracy(model, g_a, rev_a, vocab.bos_a, vocab.eos,
                                        n_samples=n_samples, cut=cut, ref_grammar_for_prefix=g_a, rng=rng)
    swapped = eval_generation_accuracy(model, g_a, rev_a, vocab.bos_b, vocab.eos,
                                        n_samples=n_samples, cut=cut, ref_grammar_for_prefix=g_a, rng=rng)
    return {'correct_tag_acc': correct['accuracy'], 'swapped_tag_acc': swapped['accuracy'],
            'drop': correct['accuracy'] - swapped['accuracy']}


# Example usage once bi_shared checkpoints exist (requires SAVE_ALL_CHECKPOINTS=True):
# for d_model in CAPACITY_SWEEP:
#     ckpt = os.path.join(RESULTS_DIR, "checkpoints", f"bi_shared_d{d_model}_s0.pt")
#     if os.path.exists(ckpt):
#         m = GPTRope(vocab_size=VOCAB_BI_SHARED.vocab_size, d_model=d_model,
#                      n_layers=N_LAYERS, n_heads=N_HEADS, max_seq_len=512).to(DEVICE)
#         m.load_state_dict(torch.load(ckpt, map_location=DEVICE))
#         result = tag_swap_test(m, GA, REV_A, VOCAB_BI_SHARED)
#         print(f"d_model={d_model}: {result}")
#         del m


### Diagnostic 2: Cross-grammar divergence sanity check

Confirms G_A and G_B are genuinely different distributions, not accidentally
similar (which would make sharing artificially cheap). Fits a simple n-gram
model to each grammar and cross-evaluates.


In [ ]:

def cross_grammar_divergence(g_a, g_b, order=4, n_fit=8000, n_eval=1000, seed=42):
    rng_fit_a, rng_fit_b = random.Random(seed), random.Random(seed + 1)
    rng_eval_a, rng_eval_b = random.Random(seed + 2), random.Random(seed + 3)
    counts_a = fit_ngram(g_a, order, n_fit, rng_fit_a)
    counts_b = fit_ngram(g_b, order, n_fit, rng_fit_b)
    # within-grammar CE (should be low: model matches its own data)
    ce_a_on_a = eval_ngram_crossentropy(g_a, counts_a, order, n_eval, rng_eval_a)
    ce_b_on_b = eval_ngram_crossentropy(g_b, counts_b, order, n_eval, rng_eval_b)
    return {'ce_a_on_a': ce_a_on_a, 'ce_b_on_b': ce_b_on_b,
            'note': 'Terminal ids differ between grammars by construction (independent generation), '
                    'so cross-evaluating A-fit-model on B-data is not directly meaningful here; '
                    'this check mainly confirms both grammars individually have non-trivial, '
                    'learnable structure of comparable difficulty (see Pre-check 2 for the real '
                    'difficulty-matching comparison via generation accuracy).'}

print(cross_grammar_divergence(GA, GB))


### Diagnostic 3: Per-position loss disaggregation

Breaks down held-out loss by position-in-sentence bucket, to see whether the ε
cost (where present) concentrates at particular structural positions —
analogous to Allen-Zhu's Result 5 showing NT information concentrates at
boundary positions.


In [ ]:

@torch.no_grad()
def per_position_loss(model, condition, g_a, g_b, vocab, n_batches=20, batch_size=32,
                       window=512, n_buckets=10, seed=777):
    '''Mean per-token loss, bucketed by relative position within the observed window
    (0 = start of window, 1 = end). Note this buckets by RAW window position, not by
    position-within-sentence (since windows span sentence boundaries) -- a coarse but
    cheap diagnostic; refine to track true within-sentence NT-level position if the
    coarse version shows a signal worth chasing.'''
    rng = random.Random(seed)
    bucket_loss = defaultdict(list)
    for _ in range(n_batches):
        x, y = make_batch(condition, g_a, g_b, vocab, batch_size, window, rng)
        x, y = x.to(DEVICE), y.to(DEVICE)
        with autocast_ctx():
            logits, _ = model(x)
        per_tok_loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), y.reshape(-1), reduction='none')
        per_tok_loss = per_tok_loss.view(x.shape)
        T = x.shape[1]
        for pos in range(T):
            bucket = min(n_buckets - 1, int(n_buckets * pos / T))
            bucket_loss[bucket].extend(per_tok_loss[:, pos].tolist())
    return {b: statistics.mean(v) for b, v in sorted(bucket_loss.items())}

# Example usage once a bi_shared checkpoint is loaded as `model`:
# print(per_position_loss(model, 'bi_shared', GA, GB, VOCAB_BI_SHARED))


## Notes on extending this notebook

- **Follow-up A (ε gradient via rule overlap):** parameterize `remap_terminals`-style
  construction differently — instead of two fully independent grammars, build G_B
  by copying G_A's rules and replacing a controlled fraction (top-down or
  bottom-up) with fresh ones. Sweep that fraction as a continuous ε knob.
- **Follow-up B (implicit-CFG / larger embedding conflict):** replace the 3
  terminals with 3 bags of 20-30 surface tokens each; only bag *membership*
  differs between languages for a controlled fraction of tokens. Requires
  extending `sample_string` to emit a surface token from `Categorical(bag_dist)`
  after generating the terminal-symbol sequence, and updating the CYK parser to
  map surface tokens back to their terminal symbol before validity checking.
- **Follow-up C (adding τ):** pair a low-`rules_per_nt` (easy, shortcut-rich)
  grammar against a high-`rules_per_nt` one instead of two matched-difficulty
  grammars.
- **Follow-up D (K-scaling):** extend `build_language_pair` to build K grammars
  and generalize `make_batch`/`build_vocab` from a hardcoded A/B pair to a list.
